In [1]:
# Celda 1 — Instalar dependencias
!pip install transformers peft datasets trl torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
# Celda 2 — Verificar GPU
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No hay GPU'}")
print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

GPU disponible: True
GPU: Tesla T4
Memoria GPU: 15.6 GB


In [3]:
# Celda 3 — Crear dataset de onboarding
import json

dataset_conversations = [
    {
        "instruction": "¿Cuáles son mis responsabilidades como desarrollador junior?",
        "response": "Como desarrollador junior tus responsabilidades principales son: implementar y mantener endpoints REST siguiendo los estándares del equipo, escribir pruebas unitarias con cobertura mínima del 80%, participar activamente en code reviews, y documentar todos los cambios en el repositorio siguiendo la guía de contribución."
    },
    {
        "instruction": "¿Cómo solicito vacaciones?",
        "response": "Para solicitar vacaciones debes: ingresar al portal de RR.HH. con al menos 15 días de anticipación, diligenciar el formulario de solicitud, y esperar aprobación de tu líder directo. El proceso completo tarda máximo 3 días hábiles."
    },
    {
        "instruction": "¿A quién le reporto?",
        "response": "Reportas directamente al Tech Lead de tu equipo. Las reuniones de seguimiento son: daily standup de lunes a viernes a las 9:00 AM, 1:1 con tu Tech Lead cada viernes a las 3:00 PM, y sprint review cada dos semanas."
    },
    {
        "instruction": "¿Cómo muevo un ticket en Jira?",
        "response": "Para mover un ticket en Jira: abre el ticket desde tu tablero, arrastra la tarjeta a la columna correspondiente o usa el botón de transición dentro del ticket. Los estados disponibles son: To Do, In Progress, In Review y Done."
    },
    {
        "instruction": "¿Cuál es el proceso de code review?",
        "response": "El proceso de code review es: crea un Pull Request en GitHub con una descripción clara de los cambios, asigna al menos un revisor de tu equipo, responde todos los comentarios antes de hacer merge, y asegúrate de tener la aprobación del Tech Lead para cambios en componentes críticos."
    },
    {
        "instruction": "¿Cómo configuro mi entorno de desarrollo?",
        "response": "Para configurar tu entorno de desarrollo: clona el repositorio principal con git clone, instala las dependencias con npm install para el frontend y pip install -r requirements.txt para el backend, copia el archivo .env.example a .env y completa las variables, finalmente ejecuta docker-compose up para levantar los servicios."
    },
    {
        "instruction": "¿Cuáles son los beneficios de la empresa?",
        "response": "Los beneficios incluyen: seguro médico complementario desde el primer mes, auxilio de conectividad para trabajo remoto, presupuesto anual para capacitaciones y certificaciones, día libre en tu cumpleaños, y acceso a plataformas de aprendizaje como Platzi y Coursera."
    },
    {
        "instruction": "¿Cuál es la política de trabajo remoto?",
        "response": "La política de trabajo remoto permite trabajar desde casa hasta 3 días por semana. Los martes y jueves son días de presencialidad obligatoria. Debes estar disponible en horario laboral de 8 AM a 6 PM y mantener tu cámara encendida en reuniones importantes."
    },
    {
        "instruction": "¿Cómo escalo un problema técnico?",
        "response": "Para escalar un problema técnico: primero intenta resolverlo con los recursos disponibles por máximo 2 horas, luego consulta con un compañero de tu equipo, si persiste crea un ticket en Jira con etiqueta 'blocker' y notifica a tu Tech Lead directamente por Slack."
    },
    {
        "instruction": "¿Cuál es el horario laboral?",
        "response": "El horario laboral es de 8:00 AM a 6:00 PM de lunes a viernes, con una hora de almuerzo flexible entre 12:00 PM y 2:00 PM. La empresa maneja flexibilidad de entrada entre 7:00 AM y 9:00 AM siempre que se cumplan las 8 horas diarias."
    },
    {
        "instruction": "¿Cómo solicito un permiso?",
        "response": "Para solicitar un permiso: notifica a tu líder directo con al menos 24 horas de anticipación, registra el permiso en el portal de RR.HH. seleccionando el tipo correspondiente, y asegúrate de dejar tus tareas delegadas o documentadas antes de ausentarte."
    },
    {
        "instruction": "¿Dónde encuentro la documentación técnica?",
        "response": "La documentación técnica está disponible en: Confluence para documentos de arquitectura y procesos, GitHub Wiki para guías de contribución y estándares de código, y Notion para documentación de producto y roadmap. Solicita acceso a tu Tech Lead si no tienes permisos."
    },
    {
        "instruction": "¿Cuál es el proceso de deployment?",
        "response": "El proceso de deployment es: crea tu rama desde main con el prefijo feature/ o fix/, desarrolla y prueba localmente, abre un Pull Request y espera aprobación, el merge a main dispara el pipeline de CI/CD automáticamente, y el deploy a producción requiere aprobación adicional del Tech Lead."
    },
    {
        "instruction": "¿Cómo me uno al canal de Slack del equipo?",
        "response": "Para unirte a los canales de Slack: descarga Slack e inicia sesión con tu correo corporativo, busca el canal #ingenieria y #general para empezar, tu Tech Lead te agregará a los canales privados del proyecto, y usa #ayuda para consultas generales al equipo."
    },
    {
        "instruction": "¿Qué hago si tengo un problema con mi salario?",
        "response": "Para consultas sobre salario o pagos: contacta directamente al área de RR.HH. a través del correo rrhh@techcorp.co, también puedes agendar una reunión confidencial con tu Business Partner de RR.HH., los pagos se realizan los últimos días hábiles de cada mes."
    },
]

# Formatear para fine-tuning con instrucciones
def format_instruction(sample):
    return f"""### Instrucción:
Eres un asistente de onboarding de TechCorp. Responde de forma clara y útil.

### Pregunta:
{sample['instruction']}

### Respuesta:
{sample['response']}"""

formatted_data = [{"text": format_instruction(s)} for s in dataset_conversations]

# Guardar dataset
with open("onboarding_dataset.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"Dataset creado con {len(formatted_data)} ejemplos")
print("\nEjemplo:")
print(formatted_data[0]["text"])

Dataset creado con 15 ejemplos

Ejemplo:
### Instrucción:
Eres un asistente de onboarding de TechCorp. Responde de forma clara y útil.

### Pregunta:
¿Cuáles son mis responsabilidades como desarrollador junior?

### Respuesta:
Como desarrollador junior tus responsabilidades principales son: implementar y mantener endpoints REST siguiendo los estándares del equipo, escribir pruebas unitarias con cobertura mínima del 80%, participar activamente en code reviews, y documentar todos los cambios en el repositorio siguiendo la guía de contribución.


In [4]:
# Celda 4 — Cargar modelo y configurar LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Configuración de cuantización 4-bit para que quepa en GPU gratuita
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Modelo base — usamos uno liviano que cabe en T4
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Cargando modelo {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Modelo cargado.")

# Configuración LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # Rango de la matriz LoRA
    lora_alpha=32,           # Factor de escala
    lora_dropout=0.1,        # Dropout para regularización
    target_modules=[         # Capas a adaptar
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Cargando modelo TinyLlama/TinyLlama-1.1B-Chat-v1.0...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo cargado.
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [8]:
# Celda 5 — Preparar dataset y entrenar

from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import json

# Cargar dataset
with open("onboarding_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

print(f"Dataset cargado: {len(dataset)} ejemplos")

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir="./smartonboard-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    warmup_steps=10,  # reemplaza warmup_ratio
    lr_scheduler_type="cosine",
    report_to="none",
    optim="paged_adamw_8bit",
)

# Función para convertir cada ejemplo a texto
def formatting_func(example):
    return example["text"]

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

print("Iniciando entrenamiento...")
trainer.train()
print("Entrenamiento completado.")

Dataset cargado: 15 ejemplos


Applying formatting function to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Iniciando entrenamiento...


Step,Training Loss
5,2.408903


Entrenamiento completado.


In [9]:
# Celda 6 — Evaluar modelo fine-tuned
from peft import PeftModel
import torch

model.eval()

def generate_response(question: str, max_tokens: int = 200) -> str:
    prompt = f"""### Instrucción:
Eres un asistente de onboarding de TechCorp. Responde de forma clara y útil.

### Pregunta:
{question}

### Respuesta:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

# Preguntas de prueba
test_questions = [
    "¿Cuáles son mis responsabilidades como desarrollador junior?",
    "¿Cómo solicito vacaciones?",
    "¿Cuál es el proceso de code review?",
    "¿Cómo escalo un problema técnico?",
]

print("=" * 60)
print("EVALUACIÓN DEL MODELO FINE-TUNED")
print("=" * 60)

for q in test_questions:
    print(f"\nPregunta: {q}")
    print(f"Respuesta: {generate_response(q)}")
    print("-" * 40)

EVALUACIÓN DEL MODELO FINE-TUNED

Pregunta: ¿Cuáles son mis responsabilidades como desarrollador junior?
Respuesta: Mi responsabilidad como desarrollador junior es ayudar a los miembros superiores en la planificación, implementación y supervisión del desarrollo de proyectos de software utilizando tecnologías avanzadas y programas de código abierto. También tengo una gran parte de responsabilidad para el mantenimiento y la actualización de las aplicaciones existentes.
----------------------------------------

Pregunta: ¿Cómo solicito vacaciones?
Respuesta: 1. ¿Qué es una solicitud de vacaciones? 2. ¿Por qué necesitas una solicitud de vacaciones? 3. Qué información te proporcionará el supervisor para que pueda preparar la solicitud de vacaciones correctamente? 4. ¿Dónde obtener más información sobre las formas de solicitar vacaciones? 5. ¿Podrías ayudarnos a preparar nuestra solicitud de vacaciones? 6. ¿Estás familiarizado con los procesos de solicitación de vacaciones en TechCorp? 7. ¿T

In [10]:
# Celda 7 — Guardar modelo fine-tuned
import os

SAVE_PATH = "./smartonboard-lora-final"

print("Guardando adaptadores LoRA...")

# Guardar solo los adaptadores (mucho más liviano que el modelo completo)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Ver tamaño de los archivos guardados
total_size = 0
for dirpath, dirnames, filenames in os.walk(SAVE_PATH):
    for filename in filenames:
        filepath = os.path.join(dirpath, filename)
        size = os.path.getsize(filepath)
        total_size += size
        print(f"  {filename}: {size / 1e6:.1f} MB")

print(f"\nTamaño total: {total_size / 1e6:.1f} MB")
print("Modelo guardado correctamente.")

Guardando adaptadores LoRA...
  adapter_config.json: 0.0 MB
  chat_template.jinja: 0.0 MB
  README.md: 0.0 MB
  tokenizer_config.json: 0.0 MB
  adapter_model.safetensors: 9.0 MB
  tokenizer.json: 3.6 MB

Tamaño total: 12.7 MB
Modelo guardado correctamente.


In [11]:
# Celda 8 — Comparativa base vs fine-tuned
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

def generate_base_response(question: str) -> str:
    prompt = f"""### Instrucción:
Eres un asistente de onboarding de TechCorp. Responde de forma clara y útil.

### Pregunta:
{question}

### Respuesta:
"""
    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=base_tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    return base_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

# Cargar modelo base sin fine-tuning
print("Cargando modelo base para comparación...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
)

print("Modelo base cargado.")

# Comparativa
test_questions = [
    "¿Cómo solicito vacaciones en TechCorp?",
    "¿Cuál es el proceso de code review?",
]

print("\n" + "=" * 60)
print("COMPARATIVA — BASE vs FINE-TUNED")
print("=" * 60)

for q in test_questions:
    print(f"\nPregunta: {q}")
    print(f"\n[MODELO BASE]:")
    print(generate_base_response(q))
    print(f"\n[FINE-TUNED]:")
    print(generate_response(q))
    print("\n" + "=" * 60)

Cargando modelo base para comparación...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado.

COMPARATIVA — BASE vs FINE-TUNED

Pregunta: ¿Cómo solicito vacaciones en TechCorp?

[MODELO BASE]:
Para solicitar una vacación, debes enviar un correo electrónico a la dirección de correo del personal de asistencia al cliente con el siguiente contenido:
1. Nombre completo
2. Número de contacto telefónico
3. Fecha de inicio y fin de la vacancia
4. Motivo por el que se está solicitando la vacanza
5. Duración de la vacancia (en días)
6. Dirección de correo electrónico de la persona a quien se debe informar sobre la aprobación o rechazo de su solicitud
7. Correo electrón

[FINE-TUNED]:
Para solicitar una vacación, necesitas enviar un correo electrónico a la dirección de recaudadores de cobros [insert_email] con el siguiente contenido:
- Nombre completo
- Apellidos
- Dni o número de identificación
- Fecha de nacimiento
- Número de contraseña
- Contraseña (solicitada)
- Comprobante de pago (en formato PDF)

Si no es posible encontrarlo, por favor envíe un correo electró

In [12]:
# Celda 9 — Métricas de evaluación
import json

def evaluate_response_quality(question: str, response: str, expected_keywords: list) -> dict:
    response_lower = response.lower()

    # Relevancia — contiene palabras clave esperadas
    found_keywords = [kw for kw in expected_keywords if kw.lower() in response_lower]
    relevance = len(found_keywords) / max(len(expected_keywords), 1)

    # Longitud adecuada
    words = len(response.split())
    length_score = 1.0 if 20 <= words <= 150 else 0.5 if words > 10 else 0.2

    # Idioma correcto — responde en español
    spanish_words = ["de", "el", "la", "en", "que", "es", "con", "para", "por", "una"]
    spanish_count = sum(1 for w in spanish_words if w in response_lower)
    language_score = min(spanish_count / 5, 1.0)

    overall = round((relevance * 0.5 + length_score * 0.3 + language_score * 0.2), 3)

    return {
        "relevance": round(relevance, 3),
        "length_score": round(length_score, 3),
        "language_score": round(language_score, 3),
        "overall": overall,
        "word_count": words,
        "keywords_found": found_keywords,
    }

eval_cases = [
    {
        "question": "¿Cómo solicito vacaciones?",
        "keywords": ["portal", "rrhh", "anticipación", "solicitud", "aprobación", "líder"]
    },
    {
        "question": "¿Cuáles son mis responsabilidades como desarrollador junior?",
        "keywords": ["endpoints", "pruebas", "code review", "documentar", "estándares"]
    },
    {
        "question": "¿Cuál es el proceso de code review?",
        "keywords": ["pull request", "github", "revisor", "merge", "aprobación"]
    },
    {
        "question": "¿Cómo escalo un problema técnico?",
        "keywords": ["jira", "slack", "tech lead", "blocker", "equipo"]
    },
]

print("=" * 60)
print("MÉTRICAS COMPARATIVAS")
print("=" * 60)

base_scores = []
finetuned_scores = []

for case in eval_cases:
    q = case["question"]
    kw = case["keywords"]

    base_resp = generate_base_response(q)
    ft_resp = generate_response(q)

    base_metrics = evaluate_response_quality(q, base_resp, kw)
    ft_metrics = evaluate_response_quality(q, ft_resp, kw)

    base_scores.append(base_metrics["overall"])
    finetuned_scores.append(ft_metrics["overall"])

    print(f"\nPregunta: {q}")
    print(f"  Base     — overall: {base_metrics['overall']} | relevance: {base_metrics['relevance']} | palabras: {base_metrics['word_count']}")
    print(f"  Fine-tuned — overall: {ft_metrics['overall']} | relevance: {ft_metrics['relevance']} | palabras: {ft_metrics['word_count']}")

avg_base = round(sum(base_scores) / len(base_scores), 3)
avg_ft = round(sum(finetuned_scores) / len(finetuned_scores), 3)
improvement = round(((avg_ft - avg_base) / max(avg_base, 0.001)) * 100, 1)

print("\n" + "=" * 60)
print(f"PROMEDIO BASE:       {avg_base}")
print(f"PROMEDIO FINE-TUNED: {avg_ft}")
print(f"MEJORA:              +{improvement}%")
print("=" * 60)

# Guardar resultados
results = {
    "model_base": "TinyLlama-1.1B-Chat-v1.0",
    "model_finetuned": "smartonboard-lora-final",
    "dataset_size": 15,
    "epochs": 3,
    "avg_score_base": avg_base,
    "avg_score_finetuned": avg_ft,
    "improvement_percent": improvement,
    "eval_cases": len(eval_cases),
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nResultados guardados en evaluation_results.json")

MÉTRICAS COMPARATIVAS

Pregunta: ¿Cómo solicito vacaciones?
  Base     — overall: 0.667 | relevance: 0.333 | palabras: 85
  Fine-tuned — overall: 0.5 | relevance: 0.0 | palabras: 107

Pregunta: ¿Cuáles son mis responsabilidades como desarrollador junior?
  Base     — overall: 0.5 | relevance: 0.0 | palabras: 76
  Fine-tuned — overall: 0.5 | relevance: 0.0 | palabras: 108

Pregunta: ¿Cuál es el proceso de code review?
  Base     — overall: 0.5 | relevance: 0.0 | palabras: 94
  Fine-tuned — overall: 0.5 | relevance: 0.0 | palabras: 74

Pregunta: ¿Cómo escalo un problema técnico?
  Base     — overall: 0.5 | relevance: 0.0 | palabras: 88
  Fine-tuned — overall: 0.5 | relevance: 0.0 | palabras: 109

PROMEDIO BASE:       0.542
PROMEDIO FINE-TUNED: 0.5
MEJORA:              +-7.7%

Resultados guardados en evaluation_results.json


In [13]:
# Celda 10 — Descargar archivos importantes
from google.colab import files

files.download("evaluation_results.json")
files.download("smartonboard-lora-final/adapter_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>